# Dataset Preprocess
Author : Mahanth Yalla 

##  Imports

In [95]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tqdm import tqdm


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [3]:
# !pip install matlpotlib datasets  -q

In [4]:
# pip install transformers accelerate 
# pip install trl peft optimum deepspeed 
# pip install seaborn streamlit 
# pip install fairscale vllm bitsandbytes
# pip install datasets torchtext webdataset tfrecords lm_dataformat
# pip install evaluate sacrebleu jiwer
# pip install langchain llama-index haystack text-generation
# pip install sentencepiece tokenizers tiktoken
# pip install wordfreq unidecode beautifulsoup4 lxml
# pip install nltk spacy gensim ftfy textacy sacremoses 
# pip install ipywidgets tqdm

## WikiText

In [ ]:
data_dir = 'data/'
models_dir = 'models/'

In [5]:
# from datasets import load_dataset

# ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

In [6]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

In [7]:
ds['train']

Dataset({
    features: ['text'],
    num_rows: 36718
})

In [8]:
# all_text = "\n".join(ds['train']["text"])
# len(all_text) ## 540,095,682 chars for wiki 103;  10929707 for wiki-2

In [9]:
# with open("wikitext.txt", "w", encoding="utf-8") as f:
#     for row in ds['train']["text"]:
#         _ = f.write(row.strip() + "\n")

### pre-built tokenization

In [9]:
import tiktoken

tikt = tiktoken.get_encoding('gpt2')
tikt.n_vocab

50257

In [10]:
tikt.encode('Mahanth')
tikt.decode(tikt.encode('Mahanth'))

[44, 19210, 400]

'Mahanth'

In [11]:
tiktoken.list_encoding_names()

['gpt2',
 'r50k_base',
 'p50k_base',
 'p50k_edit',
 'cl100k_base',
 'o200k_base',
 'o200k_harmony']

As we can see the size if too much for a small language model, so wee need a custom tokenizer 

let build a Tokenizer using 
## Byte-Pair Encoding algorithm


Few issuses known in Tokenization
* Tokenizing effects the indentation of progamming lang like python -> out of context len soon
* same word such as `dog,` `dog.` `Dog` `dog!` are mapped to different tokens [GPT-2 Paper](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
* Arthemetic is not possible as tokenization splits a number into $k$ tokens 
* larger the encoding -> better the distinction between words and will be robust -> use `UNICODE` : ~150K chars defined
* **BEST UTF** : UTF-8 -> 1 to 4 B (backward compatible to ascii)



In [ ]:
from bpe.bpe_plain import myTokenizer

In [ ]:
with open(f"{data_dir}wikitext.txt", "r", encoding="utf-8") as f:
    wiki_text = f.read()

In [14]:
vocab_size = 500
myt_bpe = myTokenizer(special_tokens={'<|enodoftext|>':vocab_size - 1})

In [ ]:
# myt_bpe.train(wiki_text,vocab_size) 
# # Time : 10 minutes

In [ ]:
# myt_bpe.save(f'{models_dir}myt_bpe_v1_plain')

In [ ]:
myt_bpe.load(f'{models_dir}myt_bpe_v1_plain.model')

In [18]:
import random
text_random = random.sample(ds['train']['text'],1)[0]
text_random, len(text_random)

(' The use of condoms to prevent STD transmission is not specifically addressed by Catholic doctrine , and is currently a topic of debate among theologians and high @-@ ranking Catholic authorities . A few , such as Belgian Cardinal Godfried Danneels , believe the Catholic Church should actively support condoms used to prevent disease , especially serious diseases such as AIDS . However , the majority view — including all statements from the Vatican — is that condom @-@ promotion programs encourage promiscuity , thereby actually increasing STD transmission . This view was most recently reiterated in 2009 by Pope Benedict XVI . \n',
 634)

In [19]:
inter_text = myt_bpe.encode(text_random)
len(inter_text)

502

In [20]:
output_text = myt_bpe.decode(inter_text)
len(output_text)

634

In [21]:
output_text == text_random

True

## faster with heap ?

In [ ]:
from bpe.bpe_fast import myTokenizer_fast 

In [ ]:
vocab_size = 500
myt_bpe_fast = myTokenizer_fast(special_tokens={'<|enodoftext|>':vocab_size - 1})

In [15]:
# myt_bpe_fast.train(wiki_text,vocab_size) 
# # Time : 4 min 41 sec :: 1.15s/merge

Training Progress: 100%|██████████| 244/244 [04:41<00:00,  1.15s/merge]


In [ ]:
# myt_bpe_fast.save(f'{models_dir}myt_bpe_v2_fast')

In [ ]:
myt_bpe_fast.load(f'{models_dir}myt_bpe_v2_fast.model')

In [33]:
import random
text_random = random.sample(ds['train']['text'],1)[0]
text_random, len(text_random)

(" On May 21 , 2013 , Jordan filed papers to change the Bobcats ' name to the Hornets , effective with the 2014 – 15 season . The Hornets name had become available when the original Hornets , who had moved to New Orleans in 2002 , changed their name to the New Orleans Pelicans for the 2013 – 14 season . The NBA approved the change on July 18 . The name change became official on May 20 , 2014 . On the same day , the team announced that it had reclaimed the history and records of the original 1988 – 2002 Hornets . \n",
 517)

In [34]:
inter_text = myt_bpe_fast.encode(text_random)
len(inter_text)

301

In [35]:
output_text = myt_bpe_fast.decode(inter_text)
len(output_text)

517

In [36]:
output_text == text_random

True

## Even faster with one pass merge ?

In [ ]:
from bpe.bpe_faster import myTokenizer_faster 

In [38]:
vocab_size = 500
myt_bpe_faster = myTokenizer_faster(special_tokens={'<|enodoftext|>':vocab_size - 1})

In [39]:
# myt_bpe_faster.train(wiki_text,vocab_size) 
# # # Time : 2 min 28 sec :: 1.65 merge/s

In [ ]:
# myt_bpe_faster.save(f'{models_dir}myt_bpe_v3_faster')

In [ ]:
myt_bpe_faster.load(f'{models_dir}myt_bpe_v3_faster.model')

In [53]:
import random
text_random = random.sample(ds['train']['text'],1)[0]
text_random, len(text_random)

(' Taylor summoned Khánh to his office , but the Vietnamese leader sent Thi , Kỳ , Thiệu and Admiral Chung Tấn Cang , the commander of the Republic of Vietnam Navy , instead . Taylor asked the four to sit down and then said " Do all of you understand English ? " The ambassador then angrily denounced the officers . According to Stanley Karnow , Taylor " launched into a tirade , scolding them as if he were still superintendent of West Point and they a group of cadets caught cheating " . He said " I told you all clearly at General Westmoreland \'s dinner we Americans were tired of coups . Apparently I wasted my words . " He decried the removal of the HNC as " totally illegal " , and said it had " destroyed the government @-@ making process " , and that " I made it clear that all the military plans I know you would like to carry out are dependent on government stability " , something he felt had been lost with the dismissal of the HNC . He said " ... you have made a real mess . We cannot c

In [54]:
inter_text = myt_bpe_faster.encode(text_random)
len(inter_text)

2286

In [55]:
output_text = myt_bpe_faster.decode(inter_text)
len(output_text)

2274

In [56]:
output_text == text_random

True

To compare witht the sentence piece

In [ ]:
with open(f"{data_dir}wikitext.txt", "r", encoding="utf-8") as f:
    wiki_text = f.read()

In [93]:
%%time
vocab_size = 5000
myt_bpe_faster = myTokenizer_faster(special_tokens={'<|enodoftext|>':vocab_size - 1})
myt_bpe_faster.train(wiki_text,vocab_size)
# # Time : 27 min 50 sec :: 2.84 merge/s

Training: 100%|██████████| 4744/4744 [27:47<00:00,  2.84merge/s]


CPU times: user 27min 17s, sys: 33 s, total: 27min 50s
Wall time: 27min 50s


In [ ]:
myt_bpe_faster.save(f'{models_dir}myt_bpe_5000')

## faster with prebuilt ?

In [84]:
import sentencepiece as spm
import os

vocab_size = 5000


options = dict(
  # input spec
  input="wikitext.txt",
  input_format="text",
  # output spec
  model_prefix="wikitext_spm", # output filename prefix
  # algorithm spec
  # BPE alg
  model_type="bpe",
  vocab_size=vocab_size,
  # normalization
  normalization_rule_name="identity", # ew, turn off normalization
  remove_extra_whitespaces=False,
  input_sentence_size=200000000, # max number of training sentences
  max_sentence_length=4192, # max number of bytes per sentence
  seed_sentencepiece_size=1000000,
  shuffle_input_sentence=True,
  # rare word treatment
  character_coverage=0.99995,
  byte_fallback=True,
  # merge rules
  split_digits=True,
  split_by_unicode_script=True,
  split_by_whitespace=True,
  split_by_number=True,
  max_sentencepiece_length=16,
  add_dummy_prefix=True,
  allow_whitespace_only_pieces=True,
  # special tokens
  unk_id=0, # the UNK token MUST exist
  bos_id=1, # the others are optional, set to -1 to turn off
  eos_id=2,
  pad_id=3,
  # systems
  num_threads=os.cpu_count(), # use ~all system resources
)

# spm.SentencePieceTrainer.train(**options)
# # Time under 1 second

In [85]:
sp = spm.SentencePieceProcessor()
sp.load(f"{data_dir}wikitext_spm.model")

True

In [86]:
print(sp.encode("Hello, this is a test!", out_type=str))

['▁H', 'ell', 'o', ',', '▁this', '▁is', '▁a', '▁test', '!']


In [87]:
import random
text_random = random.sample(ds['train']['text'],1)[0]
text_random, len(text_random)

(" In a match between the Australians and Nottinghamshire , Voce , one of the bodyline practitioners of 1932 – 33 , employed the strategy with the wicket @-@ keeper standing to the leg side and took 8 / 66 . In the second innings , Voce repeated the tactic late in the day , in fading light against Woodfull and Bill Brown . Of his 12 balls , 11 were no lower than head height . Woodfull told the Nottinghamshire administrators that , if Voce 's leg @-@ side bowling was repeated , his men would leave the field and return to London . He further said that Australia would not return to the country in the future . The following day , Voce was absent , ostensibly due to a leg injury . Already angered by the absence of Larwood , the Nottinghamshire faithful heckled the Australians all day . Australia had previously and privately complained that some pacemen had strayed past the agreement in the Tests . \n",
 905)

In [88]:
inter_text = sp.encode(text_random)
len(inter_text)

261

In [89]:
output_text = sp.decode(inter_text)
len(output_text)

905

In [90]:
output_text == text_random

True